### **Installations, Imports and Configurations**

In [1]:
# ============================================================
# Cell 1 — Check existing packages, then install only missing/outdated ones
# ============================================================

import sys
import subprocess
import importlib.metadata as importlib_metadata

# Make sure packaging exists for version comparison
try:
    from packaging.version import Version
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "packaging"])
    from packaging.version import Version


REQUIRED_PACKAGES = {
    "transformers": "4.53.0",
    "datasets": "2.20.0",
    "accelerate": "0.33.0",
    "peft": "0.13.0",
    "trl": "0.12.0",
    "bitsandbytes": "0.43.0",
    "sacrebleu": "2.4.0",
    "sentencepiece": None,
    "pandas": None,
    "tqdm": None,
    "huggingface_hub": None,
}


def get_installed_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


packages_to_install = []

print("=" * 70)
print("Checking required packages")
print("=" * 70)

for package_name, min_version in REQUIRED_PACKAGES.items():
    installed_version = get_installed_version(package_name)

    if installed_version is None:
        print(f"[MISSING]  {package_name}")
        if min_version is None:
            packages_to_install.append(package_name)
        else:
            packages_to_install.append(f"{package_name}>={min_version}")

    elif min_version is not None and Version(installed_version) < Version(min_version):
        print(f"[OLD]      {package_name}: installed={installed_version}, required>={min_version}")
        packages_to_install.append(f"{package_name}>={min_version}")

    else:
        if min_version is None:
            print(f"[OK]       {package_name}: installed={installed_version}")
        else:
            print(f"[OK]       {package_name}: installed={installed_version}, required>={min_version}")


if packages_to_install:
    print("\n" + "=" * 70)
    print("Installing/upgrading packages")
    print("=" * 70)
    print(packages_to_install)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U"] + packages_to_install
    )

    print("\nInstallation finished.")
    print("Important: if Colab asks to restart the runtime, restart it before continuing.")
else:
    print("\nAll required packages already satisfy the requirements.")

Checking required packages
[OK]       transformers: installed=5.9.0, required>=4.53.0
[OK]       datasets: installed=4.0.0, required>=2.20.0
[OK]       accelerate: installed=1.13.0, required>=0.33.0
[OK]       peft: installed=0.19.1, required>=0.13.0
[MISSING]  trl
[MISSING]  bitsandbytes
[MISSING]  sacrebleu
[OK]       sentencepiece: installed=0.2.1
[OK]       pandas: installed=2.2.2
[OK]       tqdm: installed=4.67.3
[OK]       huggingface_hub: installed=1.17.0

Installing/upgrading packages
['trl>=0.12.0', 'bitsandbytes>=0.43.0', 'sacrebleu>=2.4.0']

Installation finished.
Important: if Colab asks to restart the runtime, restart it before continuing.


In [3]:
# ============================================================
# Cell 2 — Mount Drive and create new experiment folder
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import os
import gc
import zipfile
from collections import defaultdict

PROJECT_ROOT = Path("/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline")

EXPERIMENT_NAME = "nile_chat4b_all_dialects_official_prompt_lora_r4_v1"

RUN_ROOT = PROJECT_ROOT / EXPERIMENT_NAME
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
EVAL_DIR = RUN_ROOT / "evaluation"
SUBMISSION_DIR = RUN_ROOT / "submissions"
SCORE_DIR = RUN_ROOT / "scores"
LOG_DIR = RUN_ROOT / "logs"

for d in [PROJECT_ROOT, RUN_ROOT, CHECKPOINT_DIR, EVAL_DIR, SUBMISSION_DIR, SCORE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline
RUN_ROOT: /content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/nile_chat4b_all_dialects_official_prompt_lora_r4_v1
CHECKPOINT_DIR: /content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/nile_chat4b_all_dialects_official_prompt_lora_r4_v1/checkpoints


In [4]:
# # ============================================================
# # Cell 3 — Optional Hugging Face login
# # ============================================================

# from huggingface_hub import notebook_login

# # Run this if loading the model fails because of access/gated license.
# notebook_login()

In [5]:
# ============================================================
# Cell 4 — Imports and global configuration
# ============================================================

from __future__ import annotations

import gc
import json
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import torch
import pandas as pd
from datasets import Dataset, get_dataset_config_names, get_dataset_split_names, load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# ------------------------------------------------------------
# Dataset and model
# ------------------------------------------------------------

DATASET_NAME = "UBC-NLP/alexandria"

COUNTRIES = ["EG", "JO", "LB", "LY", "MA", "MR", "OM", "PS", "SA", "SD", "SY", "TN", "YE"]

MODEL_NAME = "MBZUAI-Paris/Nile-Chat-4B"

# ------------------------------------------------------------
# Development data paths
# ------------------------------------------------------------

# Local development data should follow the organizer structure:
# development_data/
#   alexandria_*_dev_input.jsonl
#   reference_data/references.jsonl

DEVELOPMENT_DATA_DIR = PROJECT_ROOT / "development_data"
DEVELOPMENT_REFERENCE_PATH = DEVELOPMENT_DATA_DIR / "reference_data" / "references.jsonl"

RUN_DEVELOPMENT_EVALUATION = True
RUN_DEVELOPMENT_SUBMISSION = True

# ------------------------------------------------------------
# Hardware profile
# ------------------------------------------------------------

# Use "colab_t4" for normal Colab T4.
# Use "a100" only if you actually have an A100 GPU with enough VRAM.
HARDWARE_PROFILE = "colab_t4"  # options: "colab_t4", "a100"

# ------------------------------------------------------------
# Training setup
# ------------------------------------------------------------

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

USE_4BIT = True

LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.05

MAX_NEW_TOKENS = 128

if HARDWARE_PROFILE == "a100":
    PER_DEVICE_TRAIN_BATCH_SIZE = 32
    GRADIENT_ACCUMULATION_STEPS = 4
    GENERATION_BATCH_SIZE = 256

elif HARDWARE_PROFILE == "colab_t4":
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 128
    GENERATION_BATCH_SIZE = 16

else:
    raise ValueError(f"Unknown HARDWARE_PROFILE: {HARDWARE_PROFILE}")

# ------------------------------------------------------------
# Checkpointing
# ------------------------------------------------------------

SAVE_STEPS = 10
EVAL_STEPS = 10
SAVE_TOTAL_LIMIT = 80

# We do NOT select final model by eval_loss.
# Final model will be selected later by generated dev spBLEU / chrF++.
LOAD_BEST_BY_EVAL_LOSS_AT_END = False

FINAL_ADAPTER_DIR = RUN_ROOT / "final_adapter"
BEST_ADAPTER_DIR = RUN_ROOT / "best_adapter"

CHECKPOINT_SWEEP_DIR = RUN_ROOT / "checkpoint_sweep"
CHECKPOINT_SWEEP_PRED_DIR = CHECKPOINT_SWEEP_DIR / "predictions"
CHECKPOINT_SWEEP_SCORE_DIR = CHECKPOINT_SWEEP_DIR / "scores"

for d in [
    FINAL_ADAPTER_DIR,
    CHECKPOINT_SWEEP_DIR,
    CHECKPOINT_SWEEP_PRED_DIR,
    CHECKPOINT_SWEEP_SCORE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Best checkpoint selection metric
# ------------------------------------------------------------

# For EG-focused leaderboard chasing:
BEST_SELECTION_PRIMARY_METRIC = "chrfpp_EG"
BEST_SELECTION_SECONDARY_METRIC = "spbleu_EG"

# If later you want official average instead, change to:
# BEST_SELECTION_PRIMARY_METRIC = "chrfpp_avg"
# BEST_SELECTION_SECONDARY_METRIC = "spbleu_avg"

# ------------------------------------------------------------
# Save run config
# ------------------------------------------------------------

RUN_CONFIG = {
    "dataset_name": DATASET_NAME,
    "countries": COUNTRIES,
    "model_name": MODEL_NAME,
    "hardware_profile": HARDWARE_PROFILE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "use_4bit": USE_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "load_best_by_eval_loss_at_end": LOAD_BEST_BY_EVAL_LOSS_AT_END,
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "best_adapter_dir": str(BEST_ADAPTER_DIR),
    "best_selection_primary_metric": BEST_SELECTION_PRIMARY_METRIC,
    "best_selection_secondary_metric": BEST_SELECTION_SECONDARY_METRIC,
}

(RUN_ROOT / "run_config.json").write_text(
    json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

{
  "dataset_name": "UBC-NLP/alexandria",
  "countries": [
    "EG",
    "JO",
    "LB",
    "LY",
    "MA",
    "MR",
    "OM",
    "PS",
    "SA",
    "SD",
    "SY",
    "TN",
    "YE"
  ],
  "model_name": "MBZUAI-Paris/Nile-Chat-4B",
  "hardware_profile": "colab_t4",
  "max_seq_length": 1024,
  "num_epochs": 1,
  "learning_rate": 0.0002,
  "use_4bit": true,
  "lora_r": 4,
  "lora_alpha": 8,
  "lora_dropout": 0.05,
  "per_device_train_batch_size": 1,
  "gradient_accumulation_steps": 128,
  "effective_batch_size": 128,
  "generation_batch_size": 16,
  "max_new_tokens": 128,
  "save_steps": 10,
  "eval_steps": 10,
  "save_total_limit": 80,
  "load_best_by_eval_loss_at_end": false,
  "final_adapter_dir": "/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/nile_chat4b_all_dialects_official_prompt_lora_r4_v1/final_adapter",
  "best_adapter_dir": "/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/nile_chat4b_all_dialects_official_prompt_lora_r4_v1/best_adapter

### **Data Loading and Preparation**

In [6]:
# ============================================================
# Cell 5 — Development data check
# ============================================================

print("Expected development data folder:", DEVELOPMENT_DATA_DIR)
print("Expected reference path:", DEVELOPMENT_REFERENCE_PATH)

if not DEVELOPMENT_DATA_DIR.exists():
    print("development_data folder not found inside PROJECT_ROOT.")
    print("Copy the organizer development_data folder to:")
    print(DEVELOPMENT_DATA_DIR)
else:
    print("development_data exists.")

if not DEVELOPMENT_REFERENCE_PATH.exists():
    print("Reference file not found:")
    print(DEVELOPMENT_REFERENCE_PATH)
else:
    print("Reference file exists.")

input_paths = sorted(DEVELOPMENT_DATA_DIR.glob("alexandria_*_dev_input.jsonl")) if DEVELOPMENT_DATA_DIR.exists() else []
print("Development input files found:", len(input_paths))
for p in input_paths[:10]:
    print("-", p.name)

Expected development data folder: /content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/development_data
Expected reference path: /content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/development_data/reference_data/references.jsonl
development_data folder not found inside PROJECT_ROOT.
Copy the organizer development_data folder to:
/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/development_data
Reference file not found:
/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/development_data/reference_data/references.jsonl
Development input files found: 0


In [7]:
# ============================================================
# Cell 6 — Discover available HF train splits
# ============================================================

def discover_hf_splits(dataset_name: str, countries: list[str]) -> dict[str, set[str]]:
    available_configs = set(get_dataset_config_names(dataset_name))
    split_map: dict[str, set[str]] = {}

    for country in countries:
        if country not in available_configs:
            split_map[country] = set()
            continue
        split_map[country] = set(get_dataset_split_names(dataset_name, country))

    return split_map


hf_split_map = discover_hf_splits(DATASET_NAME, COUNTRIES)
TRAIN_COUNTRIES = [country for country in COUNTRIES if "train" in hf_split_map.get(country, set())]

print("Hugging Face train countries:", TRAIN_COUNTRIES)
print("Countries without HF train split:", [c for c in COUNTRIES if c not in TRAIN_COUNTRIES])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Hugging Face train countries: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
Countries without HF train split: ['LY', 'SD']


In [8]:
# ============================================================
# Cell 7 — Normalization, prompt, and pair builders
# ============================================================

def _turn_order(turn: dict, fallback: int = 0) -> int:
    try:
        return int(turn.get("turn_order", fallback))
    except (TypeError, ValueError):
        return fallback


def sorted_turns(turns: list[dict]) -> list[dict]:
    return sorted(turns or [], key=lambda turn: _turn_order(turn))


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_hf_record(row: dict) -> dict:
    english_turns = sorted_turns(row.get("english_conversation", []))
    dialect_turns = {
        _turn_order(turn, index + 1): turn
        for index, turn in enumerate(sorted_turns(row.get("dialectal_conversation", [])))
    }

    turns = []
    whole_conversation_lines = []

    for index, english_turn in enumerate(english_turns, start=1):
        order = _turn_order(english_turn, index)
        reference_turn = dialect_turns.get(order, {})
        speaker = str(english_turn.get("speaker", "")).strip()
        sentence = str(english_turn.get("text", "")).strip()

        whole_conversation_lines.append(f"{speaker}: {sentence}" if speaker else sentence)

        turns.append({
            "turn_order": order,
            "speaker": speaker,
            "sentence": sentence,
            "direction": str(english_turn.get("direction", "")).strip(),
            "reference": str(reference_turn.get("text", "")).strip(),
        })

    return {
        "conv_id": str(row.get("conv_id", "")).strip(),
        "country": str(row.get("country", "")).strip(),
        "domain": str(row.get("domain", "")).strip(),
        "dialect": str(row.get("dialect", "Arabic Dialect")).strip() or "Arabic Dialect",
        "participants": str(row.get("participants", "")).strip(),
        "whole_conversation": "\n\n".join(whole_conversation_lines),
        "turns": turns,
    }


def normalize_development_input_record(row: dict) -> dict:
    turns = []

    for index, turn in enumerate(sorted_turns(row.get("turns", [])), start=1):
        turns.append({
            "turn_order": _turn_order(turn, index),
            "speaker": str(turn.get("speaker", "")).strip(),
            "sentence": str(turn.get("sentence", turn.get("text", ""))).strip(),
            "direction": str(turn.get("direction", "")).strip(),
            "reference": "",
        })

    return {
        "conv_id": str(row.get("conv_id", "")).strip(),
        "country": str(row.get("country", "")).strip(),
        "domain": str(row.get("domain", "")).strip(),
        "dialect": str(row.get("dialect", "Arabic Dialect")).strip() or "Arabic Dialect",
        "participants": str(row.get("participants", "")).strip(),
        "whole_conversation": str(row.get("whole_conversation", "")).strip(),
        "turns": turns,
    }


def load_reference_lookup(path: Path) -> dict[tuple[str, str, int], str]:
    references = {}

    for record in read_jsonl(path):
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()

        for index, turn in enumerate(sorted_turns(record.get("turns", [])), start=1):
            turn_order = _turn_order(turn, index)
            key = (country, conv_id, turn_order)

            if key in references:
                raise ValueError(f"Duplicate reference key: {key}")

            references[key] = str(turn.get("reference", "")).strip()

    return references


def attach_references(records: list[dict], references: dict[tuple[str, str, int], str]) -> list[dict]:
    missing = []

    for record in records:
        country = record["country"]
        conv_id = record["conv_id"]

        for turn in record.get("turns", []):
            key = (country, conv_id, int(turn["turn_order"]))
            reference = references.get(key)

            if reference is None:
                missing.append(key)
                continue

            turn["reference"] = reference

    if missing:
        raise ValueError(f"Missing {len(missing)} development references; first missing keys: {missing[:5]}")

    return records


def format_history(history: list[dict]) -> str:
    if not history:
        return "No previous turns (Start of conversation)."

    lines = []

    for item in history:
        speaker = item.get("speaker", "Speaker") or "Speaker"
        lines.append(f"{speaker}: {item['sentence']}\nTranslation: {item['translation']}")

    return "\n".join(lines)


def build_prompt(record: dict, turn: dict, history: list[dict]) -> str:
    dialect = record.get("dialect") or "Arabic Dialect"
    domain = record.get("domain") or "Unknown Domain"
    participants = record.get("participants") or "Unknown Participants"
    direction = turn.get("direction") or "Unknown"
    speaker = turn.get("speaker") or "Unknown Speaker"

    return (
        f"You are an expert translator. Translate the English sentence into {dialect}.\n\n"
        f"### Metadata:\n"
        f"- Country: {record.get('country', '')}\n"
        f"- Domain: {domain}\n"
        f"- Participants: {participants}\n"
        f"- Speaker: {speaker}\n"
        f"- Speaker Direction: {direction}\n\n"
        f"### Conversation History:\n"
        f"{format_history(history)}\n\n"
        f"### Sentence to Translate:\n"
        f"{turn.get('sentence', '').strip()}\n\n"
        f"### Translation:\n"
    )


def create_finetuning_pairs(record: dict) -> list[dict]:
    pairs = []
    history: list[dict] = []

    for turn in sorted_turns(record.get("turns", [])):
        sentence = turn.get("sentence", "").strip()
        reference = turn.get("reference", "").strip()

        if not sentence or not reference:
            continue

        pairs.append({
            "prompt": build_prompt(record, turn, history),
            "response": reference,
            "country": record.get("country", ""),
            "conv_id": record.get("conv_id", ""),
            "turn_order": turn.get("turn_order"),
        })

        history.append({
            "speaker": turn.get("speaker", ""),
            "sentence": sentence,
            "translation": reference,
        })

    return pairs

In [9]:
# ============================================================
# Cell 8 — Load train and development records
# Uses HF train splits for training.
# Uses local development_data if available; otherwise falls back to HF dev splits.
# ============================================================

def load_hf_records(countries: list[str], split: str) -> list[dict]:
    records: list[dict] = []

    for country in tqdm(countries, desc=f"Loading HF {split} countries", unit="country"):
        available_splits = hf_split_map.get(country, set())

        if split not in available_splits:
            print(f"Skipping {country}: no HF {split} split.")
            continue

        dataset = load_dataset(DATASET_NAME, country, split=split)
        country_records = [normalize_hf_record(row) for row in dataset]
        records.extend(country_records)

        print(f"Loaded {len(country_records):>5} {split} conversations for {country}")

    return records


def load_development_records_from_local(data_dir: Path, reference_path: Path) -> list[dict]:
    if not data_dir.exists():
        raise FileNotFoundError(f"Development data directory does not exist: {data_dir}")

    if not reference_path.exists():
        raise FileNotFoundError(f"Development reference file does not exist: {reference_path}")

    input_paths = sorted(data_dir.glob("alexandria_*_dev_input.jsonl"))

    if not input_paths:
        raise FileNotFoundError(f"No development input files found under {data_dir}")

    records: list[dict] = []

    for path in tqdm(input_paths, desc="Loading local development input files", unit="file"):
        country_records = [normalize_development_input_record(row) for row in read_jsonl(path)]
        records.extend(country_records)
        print(f"Loaded {len(country_records):>5} development conversations from {path.name}")

    references = load_reference_lookup(reference_path)
    records = attach_references(records, references)

    return records


# ------------------------------------------------------------
# Train records: always from HF train splits
# ------------------------------------------------------------

hf_train_records = load_hf_records(TRAIN_COUNTRIES, split="train")

# ------------------------------------------------------------
# Development records:
# Prefer local organizer-style development_data if available.
# Otherwise, use HF dev splits directly.
# ------------------------------------------------------------

USE_LOCAL_DEVELOPMENT_DATA = (
    DEVELOPMENT_DATA_DIR.exists()
    and DEVELOPMENT_REFERENCE_PATH.exists()
    and len(list(DEVELOPMENT_DATA_DIR.glob("alexandria_*_dev_input.jsonl"))) > 0
)

if USE_LOCAL_DEVELOPMENT_DATA:
    print("=" * 80)
    print("Using local organizer-style development_data folder.")
    print("Development data directory:", DEVELOPMENT_DATA_DIR)
    print("Reference path:", DEVELOPMENT_REFERENCE_PATH)
    print("=" * 80)

    development_records = load_development_records_from_local(
        DEVELOPMENT_DATA_DIR,
        DEVELOPMENT_REFERENCE_PATH,
    )

    DEVELOPMENT_SOURCE = "local_development_data"

else:
    print("=" * 80)
    print("Local development_data folder was not found or incomplete.")
    print("Falling back to HF dev splits.")
    print("=" * 80)

    DEV_COUNTRIES = [
        country
        for country in COUNTRIES
        if "dev" in hf_split_map.get(country, set())
    ]

    print("HF dev countries:", DEV_COUNTRIES)

    development_records = load_hf_records(DEV_COUNTRIES, split="dev")

    DEVELOPMENT_SOURCE = "hf_dev_splits"


# ------------------------------------------------------------
# Create turn-level training/evaluation pairs
# ------------------------------------------------------------

train_pairs = [
    pair
    for record in hf_train_records
    for pair in create_finetuning_pairs(record)
]

development_pairs = [
    pair
    for record in development_records
    for pair in create_finetuning_pairs(record)
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 80)
print("Data loading summary")
print("=" * 80)

print("Development source:", DEVELOPMENT_SOURCE)
print("Train conversations:", len(hf_train_records))
print("Development conversations:", len(development_records))
print("Train turn-level pairs:", len(train_pairs))
print("Development turn-level pairs:", len(development_pairs))

train_country_counts = pd.Series([p["country"] for p in train_pairs]).value_counts().sort_index()
dev_country_counts = pd.Series([p["country"] for p in development_pairs]).value_counts().sort_index()

print("\nTrain countries:")
display(train_country_counts.to_frame("train_turns"))

print("\nDevelopment countries:")
display(dev_country_counts.to_frame("dev_turns"))

# Save data summary
DATA_SUMMARY = {
    "development_source": DEVELOPMENT_SOURCE,
    "train_conversations": len(hf_train_records),
    "development_conversations": len(development_records),
    "train_pairs": len(train_pairs),
    "development_pairs": len(development_pairs),
    "train_country_turn_counts": train_country_counts.to_dict(),
    "development_country_turn_counts": dev_country_counts.to_dict(),
}

(LOG_DIR / "data_summary.json").write_text(
    json.dumps(DATA_SUMMARY, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nSaved data summary to:")
print(LOG_DIR / "data_summary.json")

Loading HF train countries:   0%|          | 0/11 [00:00<?, ?country/s]

EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

...

Data loading summary
Development source: hf_dev_splits
Train conversations: 21146
Development conversations: 3963
Train turn-level pairs: 66480
Development turn-level pairs: 12250

Train countries:


,train_turns
EG,3108
JO,5501
LB,8906
MA,2573
MR,5515
OM,6280
PS,14933
SA,8470
SY,6071
TN,2034



Development countries:


,dev_turns
EG,1113
JO,1113
LB,1118
MA,1110
MR,1114
OM,1109
PS,1110
SA,1110
SY,1119
TN,1116



Saved data summary to:
/content/drive/MyDrive/alexandriax_nile_chat4b_official_pipeline/nile_chat4b_all_dialects_official_prompt_lora_r4_v1/logs/data_summary.json


### **Model and training preparation**

In [10]:
# ============================================================
# Cell 9 — Load tokenizer and build SFT datasets
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)


def build_sft_dataset(pairs: list[dict]) -> Dataset:
    rows = []

    for item in pairs:
        rows.append({
            "prompt": item["prompt"],
            "completion": item["response"].strip() + tokenizer.eos_token,
            "country": item["country"],
            "conv_id": item["conv_id"],
            "turn_order": item["turn_order"],
        })

    return Dataset.from_list(rows)


train_dataset = build_sft_dataset(train_pairs)
eval_dataset = build_sft_dataset(development_pairs)

print(train_dataset)
print(eval_dataset)

print("\nExample prompt:")
print(train_dataset[0]["prompt"][:2000])

print("\nExample completion:")
print(train_dataset[0]["completion"])

config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

EOS token: <eos>
PAD token: <pad>
Padding side: right
Dataset({
    features: ['prompt', 'completion', 'country', 'conv_id', 'turn_order'],
    num_rows: 66480
})
Dataset({
    features: ['prompt', 'completion', 'country', 'conv_id', 'turn_order'],
    num_rows: 12250
})

Example prompt:
You are an expert translator. Translate the English sentence into Egyptian Arabic (Cairene) Dialect.

### Metadata:
- Country: EG
- Domain: Agriculture and farming
- Participants: Wholesale Buyer, Wholesale Seller
- Speaker: Wholesale Buyer
- Speaker Direction: male -> female

### Conversation History:
No previous turns (Start of conversation).

### Sentence to Translate:
Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?

### Translation:


Example completion:
صباح الخير، عايز عشرة طن من الخرشوف الكويس للتصدير، بيقولوا ان احسن جودة في سوق العبور بتيجي من عندكم، صحيح؟<eos>


In [11]:
# ============================================================
# Cell 10 — Optional length check
# ============================================================

def estimate_lengths(dataset: Dataset, n: int | None = None) -> pd.DataFrame:
    rows = []
    size = len(dataset) if n is None else min(n, len(dataset))

    for i in tqdm(range(size), desc="Tokenizing examples for length check"):
        text = dataset[i]["prompt"] + dataset[i]["completion"]
        length = len(tokenizer(text, add_special_tokens=True)["input_ids"])
        rows.append({
            "idx": i,
            "country": dataset[i].get("country", ""),
            "length": length,
        })

    return pd.DataFrame(rows)


length_df = estimate_lengths(train_dataset, n=min(5000, len(train_dataset)))

display(length_df["length"].describe().to_frame())

too_long_ratio = (length_df["length"] > MAX_SEQ_LENGTH).mean()
print(f"Estimated > MAX_SEQ_LENGTH ratio on sample: {too_long_ratio:.2%}")

display(
    length_df.sort_values("length", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

Tokenizing examples for length check: 100%|██████████| 5000/5000 [00:03<00:00, 1323.43it/s]


,length
count,5000.000000
mean,192.452800
std,59.535635
min,89.000000
25%,141.000000
50%,184.000000
75%,232.000000
max,418.000000


Estimated > MAX_SEQ_LENGTH ratio on sample: 0.00%


,idx,country,length
0,2988,EG,418
1,2757,EG,415
2,2664,EG,404
3,3054,EG,400
4,2559,EG,397
5,1520,EG,396
6,2861,EG,395
7,1615,EG,391
8,2647,EG,387
9,3050,EG,387


In [ ]:
# ============================================================
# Cell 11 — Load model for QLoRA training
# ============================================================

def model_compute_dtype() -> torch.dtype:
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not USE_4BIT:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = model_compute_dtype()
quantization_config = build_quantization_config()

print("compute_dtype:", compute_dtype)
print("bf16_supported:", bf16_supported)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=compute_dtype,
    quantization_config=quantization_config,
    trust_remote_code=True,
)

model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

print("Model loaded for training.")
print("Model class:", model.__class__.__name__)

compute_dtype: torch.bfloat16
bf16_supported: True


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/37.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

### LoRA and SFTTrainer

In [ ]:
# ============================================================
# Cell 12 — Configure LoRA and SFTTrainer
# Compatible with TRL versions where SFTConfig does not accept save_safetensors
# ============================================================

import inspect

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

# ------------------------------------------------------------
# Build SFTConfig safely
# ------------------------------------------------------------

sft_config_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",

    # --------------------------------------------------------
    # Sequence / loss
    # --------------------------------------------------------
    "max_length": MAX_SEQ_LENGTH,
    "completion_only_loss": True,
    "packing": False,

    # --------------------------------------------------------
    # Logging / evaluation / checkpointing
    # --------------------------------------------------------
    "logging_steps": 10,

    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,

    "eval_steps": EVAL_STEPS,

    # Do not choose best by eval_loss.
    # We will select best later using generated dev spBLEU / chrF++.
    "load_best_model_at_end": False,

    # --------------------------------------------------------
    # Precision / memory
    # --------------------------------------------------------
    "bf16": bf16_supported,
    "fp16": not bf16_supported,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit" if USE_4BIT else "adamw_torch",

    # --------------------------------------------------------
    # Misc
    # --------------------------------------------------------
    "report_to": "none",
    "remove_unused_columns": True,
}

# TRL / Transformers version compatibility:
# Some versions use eval_strategy, others use evaluation_strategy.
sft_config_signature = inspect.signature(SFTConfig.__init__)
sft_config_params = set(sft_config_signature.parameters.keys())

if "eval_strategy" in sft_config_params:
    sft_config_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sft_config_params:
    sft_config_kwargs["evaluation_strategy"] = "steps"
else:
    print("Warning: neither eval_strategy nor evaluation_strategy exists in this SFTConfig.")

# Remove unsupported arguments automatically.
filtered_sft_config_kwargs = {
    key: value
    for key, value in sft_config_kwargs.items()
    if key in sft_config_params
}

removed_args = sorted(set(sft_config_kwargs) - set(filtered_sft_config_kwargs))

if removed_args:
    print("Removed unsupported SFTConfig arguments:")
    for arg in removed_args:
        print("-", arg)

training_args = SFTConfig(**filtered_sft_config_kwargs)

# ------------------------------------------------------------
# Build SFTTrainer safely
# ------------------------------------------------------------

sft_trainer_signature = inspect.signature(SFTTrainer.__init__)
sft_trainer_params = set(sft_trainer_signature.parameters.keys())

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": eval_dataset,
    "peft_config": lora_config,
}

# Newer TRL uses processing_class.
# Older TRL may use tokenizer.
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    print("Warning: neither processing_class nor tokenizer exists in this SFTTrainer.")

trainer = SFTTrainer(**trainer_kwargs)

print("=" * 80)
print("Trainer ready.")
print("=" * 80)

print("Checkpoint root:", CHECKPOINT_DIR)
print("Save every steps:", SAVE_STEPS)
print("Keep checkpoint limit:", SAVE_TOTAL_LIMIT)
print("Eval every steps:", EVAL_STEPS)
print("Effective batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

print("\nSFTConfig arguments used:")
for key in sorted(filtered_sft_config_kwargs.keys()):
    print(f"- {key}: {filtered_sft_config_kwargs[key]}")

In [ ]:
# ============================================================
# Cell 13 — Train with automatic resume from latest checkpoint
# ============================================================

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None

if CHECKPOINT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if last_checkpoint is not None:
    print("=" * 80)
    print("Found existing checkpoint. Resuming training from:")
    print(last_checkpoint)
    print("=" * 80)
else:
    print("=" * 80)
    print("No previous checkpoint found. Starting training from scratch.")
    print("=" * 80)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

# ------------------------------------------------------------
# Save final adapter after training finishes
# ------------------------------------------------------------

FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

metrics_path = LOG_DIR / "train_result.json"
metrics_path.write_text(
    json.dumps(train_result.metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("=" * 80)
print("Training finished.")
print("Saved final LoRA adapter to:")
print(FINAL_ADAPTER_DIR)
print("=" * 80)

print("Training metrics:")
print(json.dumps(train_result.metrics, indent=2, ensure_ascii=False))

# ------------------------------------------------------------
# Cleanup training objects from GPU
# ------------------------------------------------------------

del trainer
del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 13B — List saved checkpoints
# ============================================================

def list_checkpoint_dirs(checkpoint_root: Path) -> list[Path]:
    checkpoints = []

    if not checkpoint_root.exists():
        return []

    for path in checkpoint_root.glob("checkpoint-*"):
        if not path.is_dir():
            continue

        try:
            step = int(path.name.split("-")[-1])
        except ValueError:
            continue

        checkpoints.append((step, path))

    checkpoints = sorted(checkpoints, key=lambda item: item[0])

    return [path for _, path in checkpoints]


saved_checkpoints = list_checkpoint_dirs(CHECKPOINT_DIR)

print("=" * 80)
print(f"Found {len(saved_checkpoints)} saved checkpoints under:")
print(CHECKPOINT_DIR)
print("=" * 80)

for ckpt in saved_checkpoints:
    print("-", ckpt.name)

print("\nFinal adapter directory:")
print(FINAL_ADAPTER_DIR)

print("\nBest adapter directory:")
print(BEST_ADAPTER_DIR)

In [ ]:
# ============================================================
# Cell 14 — Load selected adapter for inference
# ============================================================

from transformers.trainer_utils import get_last_checkpoint

def model_compute_dtype() -> torch.dtype:
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not USE_4BIT:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


def is_valid_adapter_dir(adapter_dir: Path) -> bool:
    return adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists()


def choose_default_adapter_dir() -> Path:
    if is_valid_adapter_dir(BEST_ADAPTER_DIR):
        print("Using best adapter:")
        return BEST_ADAPTER_DIR

    if is_valid_adapter_dir(FINAL_ADAPTER_DIR):
        print("Using final adapter:")
        return FINAL_ADAPTER_DIR

    latest_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None

    if latest_checkpoint is not None:
        latest_checkpoint = Path(latest_checkpoint)

        if is_valid_adapter_dir(latest_checkpoint):
            print("Using latest checkpoint:")
            return latest_checkpoint

    raise FileNotFoundError(
        "No valid adapter found. Expected one of:\n"
        f"- {BEST_ADAPTER_DIR}\n"
        f"- {FINAL_ADAPTER_DIR}\n"
        f"- latest checkpoint under {CHECKPOINT_DIR}"
    )


def load_adapter_for_inference(adapter_dir: Path):
    adapter_dir = Path(adapter_dir)

    if not is_valid_adapter_dir(adapter_dir):
        raise FileNotFoundError(f"Invalid adapter directory: {adapter_dir}")

    print("=" * 80)
    print("Loading adapter for inference:")
    print(adapter_dir)
    print("=" * 80)

    tokenizer_source = adapter_dir if (adapter_dir / "tokenizer_config.json").exists() else MODEL_NAME

    inference_tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        trust_remote_code=True,
    )

    if inference_tokenizer.pad_token is None:
        inference_tokenizer.pad_token = inference_tokenizer.eos_token

    inference_tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=model_compute_dtype(),
        quantization_config=build_quantization_config(),
        trust_remote_code=True,
    )

    inference_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    if hasattr(inference_model, "gradient_checkpointing_disable"):
        inference_model.gradient_checkpointing_disable()

    inference_model.eval()
    inference_model.config.use_cache = True

    print("Model loaded.")
    print("Tokenizer padding side:", inference_tokenizer.padding_side)

    return inference_model, inference_tokenizer


ADAPTER_DIR_TO_LOAD = choose_default_adapter_dir()

fine_tuned_model, tokenizer = load_adapter_for_inference(ADAPTER_DIR_TO_LOAD)

### **Inference, Scoring and Evaluation**

In [ ]:
# ============================================================
# Cell 15 — Prediction helpers
# ============================================================

def count_turns(records: list[dict]) -> int:
    return sum(len(record.get("turns", [])) for record in records)


def chunks(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def clean_generation(text: str) -> str:
    text = text.strip()

    markers = [
        "\n###",
        "### Sentence to Translate:",
        "### Translation:",
        "Sentence to Translate:",
        "Translation:",
        "<turn|>",
    ]

    for marker in markers:
        if marker in text:
            text = text.split(marker, 1)[0].strip()

    return text.strip()


def generate_translations(
    prompts: list[str],
    model,
    tokenizer,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> list[str]:
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

    tokenizer.padding_side = previous_padding_side

    return [clean_generation(text) for text in decoded]


def generate_prediction_records(
    records: list[dict],
    model,
    tokenizer,
    batch_size: int = GENERATION_BATCH_SIZE,
    desc: str = "Generating predictions",
) -> list[dict]:
    histories: dict[int, list[dict]] = defaultdict(list)

    outputs = [
        {
            "conv_id": record["conv_id"],
            "country": record["country"],
            "turns": [],
        }
        for record in records
    ]

    max_turns = max((len(record.get("turns", [])) for record in records), default=0)

    with tqdm(total=count_turns(records), desc=desc, unit="turn") as progress:
        for turn_position in range(max_turns):
            active = []

            for record_index, record in enumerate(records):
                turns = sorted_turns(record.get("turns", []))

                if turn_position >= len(turns):
                    continue

                turn = turns[turn_position]
                prompt = build_prompt(record, turn, histories[record_index])
                active.append((record_index, turn, prompt))

            for batch in chunks(active, batch_size):
                prompts = [item[2] for item in batch]
                translations = generate_translations(prompts, model, tokenizer)

                for (record_index, turn, _), translation in zip(batch, translations):
                    outputs[record_index]["turns"].append({
                        "turn_order": int(turn["turn_order"]),
                        "prediction": translation,
                    })

                    histories[record_index].append({
                        "speaker": turn.get("speaker", ""),
                        "sentence": turn.get("sentence", ""),
                        "translation": translation,
                    })

                progress.update(len(batch))

    return outputs


def write_jsonl(records: list[dict], path: Path, desc: str | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)

    iterator = tqdm(records, desc=desc or f"Writing {path.name}", unit="conversation")

    with path.open("w", encoding="utf-8") as handle:
        for record in iterator:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    return path


def make_submission_zip(predictions_jsonl: Path, zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(predictions_jsonl, arcname="predictions.jsonl")

    print(f"Wrote {zip_path}")

    return zip_path

In [ ]:
# ============================================================
# Cell 16 — Official-style scoring helpers
# ============================================================

def prediction_map(prediction_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}

    for record in prediction_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()

        for index, turn in enumerate(record.get("turns", []), start=1):
            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)

            if key in mapped:
                raise ValueError(f"Duplicate prediction key: {key}")

            mapped[key] = str(turn.get("prediction", "")).strip()

    return mapped


def reference_map(reference_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}

    for record in reference_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()

        for index, turn in enumerate(sorted_turns(record.get("turns", [])), start=1):
            reference = str(turn.get("reference", "")).strip()

            if not reference:
                continue

            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)

            if key in mapped:
                raise ValueError(f"Duplicate reference key: {key}")

            mapped[key] = reference

    return mapped


def score_prediction_records(
    prediction_records: list[dict],
    reference_records: list[dict],
    output_path: Path | None = None,
    desc: str = "Scoring countries",
) -> dict:
    from sacrebleu.metrics import BLEU, CHRF

    predictions = prediction_map(prediction_records)
    references = reference_map(reference_records)

    missing = sorted(set(references) - set(predictions))
    extra = sorted(set(predictions) - set(references))

    if missing or extra:
        raise ValueError(
            f"Prediction/reference mismatch. "
            f"Missing={missing[:5]}, extra={extra[:5]}"
        )

    bleu = BLEU(tokenize="flores200", effective_order=False)
    chrf = CHRF(word_order=2)

    by_country: dict[str, list[tuple[str, str]]] = defaultdict(list)

    for key in tqdm(sorted(references), desc="Aligning predictions", unit="turn"):
        country = key[0]
        by_country[country].append((predictions[key], references[key]))

    scores: dict[str, float | int] = {}
    spbleu_values = []
    chrfpp_values = []

    for country, rows in tqdm(sorted(by_country.items()), desc=desc, unit="country"):
        hypotheses = [prediction for prediction, _ in rows]
        refs = [reference for _, reference in rows]

        spbleu = bleu.corpus_score(hypotheses, [refs]).score
        chrfpp = chrf.corpus_score(hypotheses, [refs]).score

        spbleu_values.append(spbleu)
        chrfpp_values.append(chrfpp)

        scores[f"spbleu_{country}"] = round(spbleu, 6)
        scores[f"chrfpp_{country}"] = round(chrfpp, 6)

    scores["spbleu_avg"] = round(sum(spbleu_values) / len(spbleu_values), 6)
    scores["chrfpp_avg"] = round(sum(chrfpp_values) / len(chrfpp_values), 6)
    scores["num_countries"] = len(by_country)
    scores["num_turns"] = len(references)
    scores["num_conversations"] = len({key[:2] for key in references})

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(
            json.dumps(scores, ensure_ascii=False, indent=2, sort_keys=True),
            encoding="utf-8",
        )
        print(f"Scores saved to {output_path}")

    return scores


def display_scores(scores: dict) -> None:
    country_rows = []

    for key, value in scores.items():
        if not key.startswith("spbleu_") or key == "spbleu_avg":
            continue

        country = key.removeprefix("spbleu_")

        country_rows.append({
            "country": country,
            "spbleu": value,
            "chrfpp": scores.get(f"chrfpp_{country}"),
        })

    summary_df = pd.DataFrame([{
        "spbleu_avg": scores["spbleu_avg"],
        "chrfpp_avg": scores["chrfpp_avg"],
        "countries": scores["num_countries"],
        "conversations": scores["num_conversations"],
        "turns": scores["num_turns"],
    }])

    country_df = pd.DataFrame(country_rows).sort_values("country").reset_index(drop=True)

    display(summary_df)
    display(country_df)

In [ ]:
# ============================================================
# Cell 17 — Evaluate checkpoints, select best checkpoint, and load it
# ============================================================

from transformers.trainer_utils import get_last_checkpoint

# ------------------------------------------------------------
# Helper: checkpoint listing
# ------------------------------------------------------------

def list_checkpoint_dirs(checkpoint_root: Path) -> list[Path]:
    checkpoints = []

    if not checkpoint_root.exists():
        return []

    for path in checkpoint_root.glob("checkpoint-*"):
        if not path.is_dir():
            continue

        try:
            step = int(path.name.split("-")[-1])
        except ValueError:
            continue

        checkpoints.append((step, path))

    checkpoints = sorted(checkpoints, key=lambda item: item[0])
    return [path for _, path in checkpoints]


# ------------------------------------------------------------
# Helper: adapter validation
# ------------------------------------------------------------

def is_valid_adapter_dir(adapter_dir: Path) -> bool:
    adapter_dir = Path(adapter_dir)
    return adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists()


# ------------------------------------------------------------
# Helper: dtype and quantization config
# ------------------------------------------------------------

def model_compute_dtype() -> torch.dtype:
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not USE_4BIT:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


# ------------------------------------------------------------
# Helper: load adapter for inference
# ------------------------------------------------------------

def load_adapter_for_inference(adapter_dir: Path):
    adapter_dir = Path(adapter_dir)

    if not is_valid_adapter_dir(adapter_dir):
        raise FileNotFoundError(f"Invalid adapter directory: {adapter_dir}")

    print("=" * 80)
    print("Loading adapter for inference:")
    print(adapter_dir)
    print("=" * 80)

    tokenizer_source = adapter_dir if (adapter_dir / "tokenizer_config.json").exists() else MODEL_NAME

    inference_tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        trust_remote_code=True,
    )

    if inference_tokenizer.pad_token is None:
        inference_tokenizer.pad_token = inference_tokenizer.eos_token

    inference_tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=model_compute_dtype(),
        quantization_config=build_quantization_config(),
        trust_remote_code=True,
    )

    inference_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    if hasattr(inference_model, "gradient_checkpointing_disable"):
        inference_model.gradient_checkpointing_disable()

    inference_model.eval()
    inference_model.config.use_cache = True

    print("Model loaded.")
    print("Tokenizer padding side:", inference_tokenizer.padding_side)

    return inference_model, inference_tokenizer


# ------------------------------------------------------------
# Helper: cleanup GPU
# ------------------------------------------------------------

def unload_model_from_gpu(model_obj=None):
    if model_obj is not None:
        try:
            del model_obj
        except Exception:
            pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Helper: collect checkpoint/final candidates
# ------------------------------------------------------------

def collect_adapter_candidates() -> list[Path]:
    candidates = []

    for ckpt in list_checkpoint_dirs(CHECKPOINT_DIR):
        if is_valid_adapter_dir(ckpt):
            candidates.append(ckpt)

    if is_valid_adapter_dir(FINAL_ADAPTER_DIR):
        candidates.append(FINAL_ADAPTER_DIR)

    unique_candidates = []
    seen = set()

    for path in candidates:
        resolved = str(path.resolve())
        if resolved in seen:
            continue
        seen.add(resolved)
        unique_candidates.append(path)

    return unique_candidates


# ------------------------------------------------------------
# Helper: metric sorting
# ------------------------------------------------------------

def safe_metric_value(scores: dict, metric_name: str) -> float:
    value = scores.get(metric_name, None)

    if value is None:
        return float("-inf")

    return float(value)


# ------------------------------------------------------------
# Evaluate all checkpoints
# ------------------------------------------------------------

adapter_candidates = collect_adapter_candidates()

print("=" * 80)
print(f"Found {len(adapter_candidates)} adapter candidates to evaluate.")
print("=" * 80)

for adapter_dir in adapter_candidates:
    print("-", adapter_dir)

if not adapter_candidates:
    raise FileNotFoundError(
        "No adapter candidates found. Expected checkpoints under CHECKPOINT_DIR "
        "or final adapter under FINAL_ADAPTER_DIR."
    )

checkpoint_sweep_rows = []

for adapter_dir in adapter_candidates:
    adapter_name = adapter_dir.name

    print("\n" + "=" * 80)
    print(f"Evaluating adapter: {adapter_name}")
    print("=" * 80)

    sweep_model, sweep_tokenizer = load_adapter_for_inference(adapter_dir)

    predictions = generate_prediction_records(
        development_records,
        sweep_model,
        sweep_tokenizer,
        batch_size=GENERATION_BATCH_SIZE,
        desc=f"Generating dev predictions: {adapter_name}",
    )

    predictions_path = CHECKPOINT_SWEEP_PRED_DIR / f"{adapter_name}_development_predictions.jsonl"

    write_jsonl(
        predictions,
        predictions_path,
        desc=f"Writing predictions: {adapter_name}",
    )

    scores_path = CHECKPOINT_SWEEP_SCORE_DIR / f"{adapter_name}_scores.json"

    scores = score_prediction_records(
        predictions,
        development_records,
        output_path=scores_path,
        desc=f"Scoring: {adapter_name}",
    )

    row = {
        "adapter_name": adapter_name,
        "adapter_dir": str(adapter_dir),
        "predictions_path": str(predictions_path),
        "scores_path": str(scores_path),
        **scores,
    }

    checkpoint_sweep_rows.append(row)

    print("Scores:")
    print(json.dumps(scores, indent=2, ensure_ascii=False))

    unload_model_from_gpu(sweep_model)


# ------------------------------------------------------------
# Save checkpoint sweep table
# ------------------------------------------------------------

checkpoint_sweep_df = pd.DataFrame(checkpoint_sweep_rows)

checkpoint_sweep_csv_path = CHECKPOINT_SWEEP_DIR / "checkpoint_sweep_results.csv"
checkpoint_sweep_json_path = CHECKPOINT_SWEEP_DIR / "checkpoint_sweep_results.json"

checkpoint_sweep_df.to_csv(checkpoint_sweep_csv_path, index=False)

checkpoint_sweep_json_path.write_text(
    checkpoint_sweep_df.to_json(orient="records", force_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 80)
print("Checkpoint sweep results saved to:")
print(checkpoint_sweep_csv_path)
print(checkpoint_sweep_json_path)
print("=" * 80)

display_cols = [
    "adapter_name",
    BEST_SELECTION_PRIMARY_METRIC,
    BEST_SELECTION_SECONDARY_METRIC,
    "spbleu_EG",
    "chrfpp_EG",
    "spbleu_avg",
    "chrfpp_avg",
    "num_turns",
]

display_cols = [col for col in display_cols if col in checkpoint_sweep_df.columns]

display(
    checkpoint_sweep_df[display_cols]
    .sort_values(
        by=[BEST_SELECTION_PRIMARY_METRIC, BEST_SELECTION_SECONDARY_METRIC],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Select best checkpoint
# ------------------------------------------------------------

best_row = max(
    checkpoint_sweep_rows,
    key=lambda row: (
        safe_metric_value(row, BEST_SELECTION_PRIMARY_METRIC),
        safe_metric_value(row, BEST_SELECTION_SECONDARY_METRIC),
    ),
)

best_adapter_source = Path(best_row["adapter_dir"])

print("\n" + "=" * 80)
print("Best adapter selected:")
print("Adapter:", best_row["adapter_name"])
print("Path:", best_adapter_source)
print("Primary metric:", BEST_SELECTION_PRIMARY_METRIC, best_row.get(BEST_SELECTION_PRIMARY_METRIC))
print("Secondary metric:", BEST_SELECTION_SECONDARY_METRIC, best_row.get(BEST_SELECTION_SECONDARY_METRIC))
print("=" * 80)

if BEST_ADAPTER_DIR.exists():
    shutil.rmtree(BEST_ADAPTER_DIR)

shutil.copytree(best_adapter_source, BEST_ADAPTER_DIR)

best_selection_path = RUN_ROOT / "best_checkpoint_selection.json"

best_selection_path.write_text(
    json.dumps(best_row, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Copied best adapter to:")
print(BEST_ADAPTER_DIR)

print("Saved best checkpoint selection to:")
print(best_selection_path)


# ------------------------------------------------------------
# Load best adapter for final evaluation
# ------------------------------------------------------------

if "fine_tuned_model" in globals():
    try:
        del fine_tuned_model
    except Exception:
        pass

unload_model_from_gpu()

ADAPTER_DIR_TO_LOAD = BEST_ADAPTER_DIR

fine_tuned_model, tokenizer = load_adapter_for_inference(ADAPTER_DIR_TO_LOAD)

print("=" * 80)
print("Best adapter is now loaded and ready for final evaluation.")
print("Loaded adapter:")
print(ADAPTER_DIR_TO_LOAD)
print("=" * 80)

In [ ]:
# ============================================================
# Cell 18 — Final development prediction and scoring using best loaded adapter
# ============================================================

if RUN_DEVELOPMENT_EVALUATION:
    if "fine_tuned_model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError(
            "No inference model/tokenizer found. "
            "Run Cell 17 first to select and load the best adapter."
        )

    if "ADAPTER_DIR_TO_LOAD" not in globals():
        raise RuntimeError(
            "ADAPTER_DIR_TO_LOAD is not defined. "
            "Run Cell 17 first."
        )

    FINAL_EVAL_NAME = Path(ADAPTER_DIR_TO_LOAD).name

    print("=" * 80)
    print("Running final development evaluation")
    print("Loaded adapter:")
    print(ADAPTER_DIR_TO_LOAD)
    print("=" * 80)

    development_predictions = generate_prediction_records(
        development_records,
        fine_tuned_model,
        tokenizer,
        batch_size=GENERATION_BATCH_SIZE,
        desc=f"Generating final development predictions: {FINAL_EVAL_NAME}",
    )

    development_predictions_path = write_jsonl(
        development_predictions,
        EVAL_DIR / f"development_predictions_{FINAL_EVAL_NAME}.jsonl",
        desc=f"Writing final development predictions: {FINAL_EVAL_NAME}",
    )

    latest_development_predictions_path = write_jsonl(
        development_predictions,
        EVAL_DIR / "development_predictions_final.jsonl",
        desc="Writing stable final development predictions",
    )

    development_scores = score_prediction_records(
        development_predictions,
        development_records,
        SCORE_DIR / f"development_scores_{FINAL_EVAL_NAME}.json",
        desc=f"Scoring final development predictions: {FINAL_EVAL_NAME}",
    )

    latest_scores_path = SCORE_DIR / "development_scores_final.json"

    latest_scores_path.write_text(
        json.dumps(development_scores, indent=2, ensure_ascii=False, sort_keys=True),
        encoding="utf-8",
    )

    print("=" * 80)
    print("Final development evaluation finished.")
    print("Predictions saved to:")
    print(development_predictions_path)
    print(latest_development_predictions_path)
    print("Scores saved to:")
    print(SCORE_DIR / f"development_scores_{FINAL_EVAL_NAME}.json")
    print(latest_scores_path)
    print("=" * 80)

    display_scores(development_scores)

    development_scores

else:
    print("RUN_DEVELOPMENT_EVALUATION is False. Skipping final development evaluation.")

In [ ]:
# ============================================================
# Cell 19 — Save final experiment summary
# ============================================================

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "run_root": str(RUN_ROOT),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "config": RUN_CONFIG,
    "development_scores": development_scores if "development_scores" in globals() else None,
}

summary_path = RUN_ROOT / "experiment_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved summary to:", summary_path)

if "development_scores" in globals():
    print(json.dumps(development_scores, indent=2, ensure_ascii=False))